# Usar formato tipo Chat (System / User / Assistant)

En lugar de:

```
prompt = "List the 7 wonders of the world"
```
Formalmente existen tres roles principales en una estructura conversacional:

| Rol         | Función técnica                                                      |
| ----------- | -------------------------------------------------------------------- |
| `system`    | Define comportamiento global, reglas, estilo y contexto persistente. |
| `user`      | Entrada del usuario (consulta o instrucción).                        |
| `assistant` | Respuestas generadas por el modelo (o ejemplos en few-shot).         |

Internamente el llm hace esto

```
<|system|>
You are ...
<|user|>
Explain ECG filtering
<|assistant|>
...
```
Con la librería oficial:

In [3]:
from ollama import chat

response = chat(
    model="tinyllama",
    messages=[
        {
            "role": "system",
            "content": "You are a historian specialized in world heritage sites. Provide structured, concise, academically accurate answers."
        },
        {
            "role": "user",
            "content": "List the 7 wonders of the modern world"
        }
    ],
    options={
        "temperature": 0.6,
        "num_predict": 800
    }
)

print(response.message.content)

1. The Great Wall of China (China) - This ancient wall stretches over 21,000 miles and was built over a period of more than 2,000 years to protect China from invaders.

2. The Pyramids of Giza (Egypt) - These massive stone structures were constructed by the pharaohs in ancient Egypt between 2569 and 2449 BC for burial purposes.

3. Machu Picchu (Peru) - Located high in the Andes Mountains, this historic site was built during the Inca Empire's reign as a center of agriculture, trade, and religious worship.

4. The Colosseum (Rome) - Built by the Roman Emperor Vespasian in AD 80-81, this amphitheater served as a venue for gladiator fights, public spectacles, and other games for over 50 years.

5. The Great Barrier Reef (Australia) - This underwater wonder is made up of more than 2,300 coral reefs and over 900 islands located in the Coral Sea. It is home to an estimated 1400 species of fish and over 250 species of marine life.

6. The Great Barrier Reef (Australia) - This underwater wonde

In [4]:
from ollama import chat

response = chat(
    model="tinyllama",
    messages=[
        {
            "role": "system",
            "content": "You are a historian specialized in world heritage sites. Provide structured, concise, academically accurate answers."
        },
        {
            "role": "user",
            "content": "List only the name of the 7 wonders of the modern world"
        }
    ],
    options={
        "temperature": 0.6,
        "num_predict": 800
    }
)

print(response.message.content)

1. The Great Wall of China
2. The Pyramids of Giza
3. The Taj Mahal
4. The Colosseum
5. Machu Picchu
6. The Great Barrier Reef
7. The Sydney Opera House


In [20]:
from ollama import chat

response = chat(
    model="tinyllama",
    messages=[
        {
            "role": "system",
            "content": """You are a historian specialized in world heritage sites. 
                        When listing the 7 wonders of the modern world, include only their names without any additional information or descriptions.
                        """
        },
        {
            "role": "user",
            "content": "List the 7 wonders of the modern world"
        }
    ],
    options={
        "temperature": 0.2,
        "num_predict": 800,
        "top_p": 0.9,
    }
)

print(response.message.content)

1. The Great Wall of China
2. The Taj Mahal
3. The Pyramids of Giza, Egypt
4. The Colosseum in Rome, Italy
5. Machu Picchu, Peru
6. The Sydney Opera House, Australia
7. The Leaning Tower of Pisa, Italy


# Prompt estructurado con delimitadores (control semántico fuerte)

Para tareas técnicas (ej. clasificación, extracción, generación estructurada), usa delimitadores claros:

In [25]:
context = """
You are an expert in history.
Answer strictly following the schema below.
Only response with this format, do not include any additional information or descriptions. 
In the response include the formatting as shown in the example.

### OUTPUT FORMAT
- Title: 
- Location: 
- Historical period: 
- Technical significance: 
"""

question = "Describe Machu Picchu"

response = chat(
    model="tinyllama",
    messages=[
        {"role": "system", "content": context},
        {"role": "user", "content": question}
    ],
    options={"temperature": 0.2, "num_predict": 800, "top_p": 0.9}
)

print(response.message.content)

Machu Picchu, also known as the "lost city of the Incas," is a historical site located in Peru that has captivated the imagination and fascination of travelers for centuries. The ruins were first discovered by an American explorer named Hiram Bingham in 1911, and since then, it has become one of the most popular tourist destinations in South America.

Machu Picchu is a unique archaeological site that was built during the Inca Empire's reign (1532-1572 AD). It consists of several structures, including temples, terraces, and houses, all built into the mountainside using traditional Incan techniques. The site is surrounded by lush vegetation, providing a stunning backdrop for visitors to take in the breathtaking views.

One of the most significant features of Machu Picchu is its location, which is situated at an altitude of 2,430 meters above sea level. This makes it one of the highest points in the Andes Mountains, making it a unique and unforgettable experience for visitors. The site al

### Añadir ejemplo (few-shot fuerte)

In [26]:
context = """
You are an expert in history.

You MUST respond EXACTLY in the following format.

Example:

- Title: Example Site
- Location: Example Location
- Historical period: Example Period
- Technical significance: Example Significance

Do not add any text outside this structure.
"""

question = "Describe Machu Picchu"

response = chat(
    model="tinyllama",
    messages=[
        {"role": "system", "content": context},
        {"role": "user", "content": question}
    ],
    options={"temperature": 0.2, "num_predict": 800, "top_p": 0.9}
)

print(response.message.content)

Machu Picchu, also known as the "Lost City of the Incas," is a stunning archaeological site located in Peru's Sacred Valley. The site was built during the Inca Empire and served as a royal retreat for the emperor Pachacuti in the 15th century.

Machu Picchu is considered one of the New Seven Wonders of the World, and it has captivated visitors from all over the world with its stunning architecture, natural beauty, and historical significance. The site's location at an altitude of 2,430 meters above sea level makes it a unique and challenging destination for hikers.

The site is divided into several sections, including the Inca ruins, the Temple of Three Windows, the Sun Gate, and the Machu Picchu Museum. The Inca ruins are made up of over 200 buildings, including temples, houses, and storage facilities. The most famous structures include the Sun Gate, which leads to the main entrance to the site; the Temple of Three Windows, which is a series of three domed rooms with intricate carving

# Few-shot prompting (ejemplos explícitos)

Cuando quieres comportamiento muy controlado:

In [45]:
messages = [
    {
        "role": "system",
        "content": "You are a scientific assistant that answers only in JSON format."
    },
    {
        "role": "user",
        "content": "Describe the Eiffel Tower"
    },
    {
        "role": "assistant",
        "content": '{"name":"Eiffel Tower","country":"France","type":"Monument"}'
    },
    {
        "role": "user",
        "content": "Describe the pisa tower"
    }
]

response = chat(
    model="tinyllama",
    messages=messages,
    options={
        "temperature": 0.2,
        "top_p": 0.8,
        "repeat_penalty": 1.1,
        "num_predict": 1200,
        "seed": 42
    }
)

print(response.message.content)

Pisa Tower, also known as the Pisa Leaning Tower of Pisa, is a famous landmark located in the Italian city of Pisa. It stands at an impressive height of 538 feet (164 meters) and is one of the most iconic structures in Italy. The tower was built in the 12th century as a bell tower for St. Nicholas Church, but it quickly became known for its unique leaning due to the weight of the building on one side. Over time, engineers have attempted to stabilize the tower, but it remains a testament to human ingenuity and engineering skill. Visitors can climb the tower's 365 steps to enjoy panoramic views of Pisa and its surrounding area.


# Inyección de contexto externo (RAG simple manual)

https://www.nytimes.com/2026/02/25/technology/nvidia-earnings.html

In [47]:
document_context = """
Nvidia’s profit for the last 12 months hit $120 billion, the chip giant said on Wednesday, providing ample evidence that it is cashing in on the tech industry’s artificial intelligence boom more than any other company in the world.

Only a handful of companies, including Alphabet, Microsoft and Apple, have made a profit as much as $100 billion in a year. And none of them have grown as quickly. Just three years ago, Nvidia’s profit was $4.4 billion.

Nvidia controls about 90 percent of the market for the cutting-edge semiconductors that power A.I. projects. More than anyone else, it will benefit from the plans of Google, Amazon, Microsoft and Meta to spend more than half a trillion dollars this year building A.I. data centers.

That spending is starting to unsettle Wall Street, and Nvidia’s share price has been relatively flat in recent months. But the financial results reported by the company on Wednesday showed that demand for Nvidia’s chips was still growing at an astonishing rate.

"""

prompt = f"""
Use ONLY the information below to answer.

CONTEXT:
{document_context}

QUESTION:
what was the profit of Nvidia in the last 12 months?"""


response = chat(model='tinyllama', messages=[
  {
    'role': 'user',
    'content': prompt,
  },
])
print(response.message.content)

The profits reported by Nvidia during the previous 12 months were $120 billion, providing evidence that it is one of the few companies worldwide capable of making a profit in excess of $100 billion as measured in a year. The company also controlled about 90% of the market for cutting-edge semiconductors used to power A.I. Projects, indicating its strong position within the sector. Nvidia's share price has been relatively flat in recent months but Wall Street seems to be warming up to the tech industry's artificial intelligence boom thanks to a growing spend from Google, Amazon, Microsoft and Meta.


# Meta-prompting (control de razonamiento)

In [48]:
system_prompt = """
You are a rigorous scientific assistant.

Follow this reasoning structure:
1. Define the concept.
2. Provide historical background.
3. Provide technical explanation.
4. Conclude with implications.
"""

response = chat(model='tinyllama', messages=[
  {
    'role': 'system',
    'content': system_prompt,
  },
  {
    'role': 'user',
    'content': "Explain the concept of photosynthesis",
  }
])
print(response.message.content)

Photosynthesis is a complex process that allows plants, algae, and some bacteria to convert light energy into chemical energy, known as ATP (adenosine triphosphate), which can then be used for cellular respiration and other metabolic processes. The process of photosynthesis involves several stages:

1. Light Reception - Plants and algae have specialized structures called pigments that are responsible for detecting light and transmitting it to the chlorophyll (the part of the pigment that absorbs sunlight).

2. Reduction of Carbon Dioxide - The carbon dioxide in the air is converted into glucose (a simple sugar) by photosystem I, a specialized enzyme complex consisting of chlorophyll and two subunits called PSI-I (photosystem I isomerase II).

3. Isotopic Exchange - The photons of light that are absorbed by chlorophyll and carried to the red and blue parts of the photosynthetic electron acceptor (which is also chlorophyll) are exchanged for a more stable electron (called an Ion), which 

# RETO

Crear un chatbot en gradio que pueda explicarme una noticia elegida por el usuario.